# TF-IDF Feature Creation

In [ ]:
import sys
sys.path.append("..")
import glob
import os
from helpers.data_prep_and_print import print_df
from pyspark.ml.feature import HashingTF, Tokenizer, CountVectorizer, IDF 
from pyspark.sql import SparkSession, Window

In [ ]:
# Create local StreamingContext with a batch interval of 10s
spark = (SparkSession.builder
           .appName("TF-IDF Creation")
           .getOrCreate())

In [ ]:
# read the labeled articles from labeled_articles.csv and check for null values in the sentence column
df = spark.read.csv("../data/labeled_articles.csv", header=True, inferSchema=True)
print_df(df, 5)

## Build the TF-IDF

In [ ]:
# Tokenize the sentences into words and create a TF-IDF feature vector
tokenizer = Tokenizer(inputCol="sentence", outputCol="words")
tokenized_df = tokenizer.transform(df)
countVectorizer = CountVectorizer(inputCol="words", outputCol="rawFeatures", minDF=2.0).fit(tokenized_df)
featurizedData = countVectorizer.transform(tokenized_df)
# Fit the IDF model and transform the data to get the TF-IDF features
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurizedData)
rescaledData = idfModel.transform(featurizedData)
print_df(rescaledData,2)

In [ ]:
from pyspark.sql.functions import col
idx = countVectorizer.vocabulary.index("description")

row = (
    rescaledData
    .where(
        (col("filename") == "arxiv_annotate10_7_1.txt")
        & col("sentence").startswith("The Minimum Description Length")
    )
    .first()
)


print("TF:", row["rawFeatures"][idx])
print("IDF:", idfModel.idf[dx])
print("TF-IDF:", row["features"][idx])

In [ ]:
spark.stop()